# 09. Motion Attribution Test

Pipeline step ⑦. Verifies `attribute_motion()` on annotated, normalized pose data.

Motion attribution checks — per rep — whether the limb that moved most matches the
declared exercise pattern (`bilateral` / `alternating`). It adds metadata columns only;
it never modifies coordinates.

Pipeline position: Phase Segmentation → **Motion Attribution** → Feature Extraction

Activation by laterality:

```
bilateral_symmetric  → skipped (no active-side concept)
alternating          → per-rep active-side check
```

This notebook assumes that the following notebooks are already passing:
- 00_environment_check through 08_phase_segmentation_test

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json

import pandas as pd

from movement.annotation import apply_annotation, load_annotation_csv
from movement.config import LANDMARKS, make_coordinate_columns, make_required_columns, make_visibility_columns
from movement.exercise_definition import load_exercise_definition
from movement.io import load_pose_csv
from movement.motion_attribution import AttributionThresholds, attribute_motion
from movement.normalization import normalize_pose_by_hip_torso
from movement.pipeline import (
    AnnotationConfig, ExerciseDefinitionConfig, MotionAttributionConfig,
    NormalizationConfig, PhaseSegmentationConfig, PipelineConfig, ValidationConfig,
    run_pipeline,
)
from movement.validation import run_basic_validation

print('imports OK')

## Data Setup

Runs pipeline ①–⑥ to produce normalized + phase-labeled data ready for ⑦.

In [ ]:
csv_path = '../data/pose/sample/mediapipe_squat_synthetic.csv'
ann_path = '../data/pose/sample/mediapipe_squat_synthetic_annotation.csv'
def_dir  = '../data/definitions/exercises'

df_raw = load_pose_csv(csv_path)
run_basic_validation(
    df=df_raw,
    required_columns=make_required_columns(LANDMARKS),
    coordinate_columns=make_coordinate_columns(LANDMARKS),
    visibility_columns=make_visibility_columns(LANDMARKS),
)
ann_df    = load_annotation_csv(ann_path)
df_ann, _ = apply_annotation(df_raw, ann_df)
exercise_def = load_exercise_definition(exercise_id='squat', definitions_dir=def_dir)
df_norm, _   = normalize_pose_by_hip_torso(df=df_ann, landmarks=LANDMARKS)

print(f'exercise: {exercise_def.exercise_id}')
print(f'laterality: {exercise_def.classification["laterality"]}')
print(f'pattern in data: {df_ann["pattern"].dropna().unique().tolist()}')

## Direct attribute_motion() Test

Squat is `bilateral_symmetric` → the module should skip per-rep attribution
and return the dataframe unchanged (only None-valued attribution columns added).

In [ ]:
thresholds = AttributionThresholds(active=0.70, ambiguous=0.55, swap=0.85)

df_attr, attr_report = attribute_motion(
    df=df_norm,
    exercise_definition=exercise_def,
    thresholds=thresholds,
    mode='conservative',
)

print(f'output shape: {df_attr.shape}')
print(f'attribution report keys: {list(attr_report.as_dict().keys())}')

## Check 1: Bilateral Exercise → Module Skipped

In [ ]:
d = attr_report.as_dict()
print(json.dumps(d, indent=2, default=str))
assert d['pattern'] == 'bilateral_symmetric' or d.get('skipped') is True or d.get('num_reps_processed', 0) == 0, \
    'bilateral_symmetric exercise should be skipped'
print('PASS: bilateral_symmetric exercise → attribution skipped')

## Check 2: Output Columns Present

In [ ]:
expected_cols = [
    'detected_active_limb', 'expected_active_limb',
    'attribution_consistent', 'attribution_confidence', 'attribution_action',
]
for col in expected_cols:
    assert col in df_attr.columns, f'missing column: {col}'
print(f'PASS: all {len(expected_cols)} attribution columns present')

# Bilateral → all attribution values should be None/NaN
for col in expected_cols:
    n_nonnull = df_attr[col].notna().sum()
    print(f'  {col}: {n_nonnull} non-null frames (expected 0 for bilateral)')
    assert n_nonnull == 0, f'{col} should be null for bilateral_symmetric'

## Check 3: Coordinates Unchanged

In [ ]:
import numpy as np
coord_cols = [c for c in df_norm.columns if c.endswith(('_norm_x', '_norm_y', '_norm_z'))]
for col in coord_cols[:5]:   # spot-check first 5
    assert (df_attr[col].values == df_norm[col].values).all(), f'{col} was modified'
print(f'PASS: coordinates unchanged after attribution ({len(coord_cols)} columns spot-checked)')

## Check 4: Pipeline Integration

In [ ]:
cfg = PipelineConfig()
cfg.validation          = ValidationConfig(enabled=True)
cfg.annotation          = AnnotationConfig(enabled=True, path=ann_path)
cfg.exercise_definition = ExerciseDefinitionConfig(enabled=True,
                              definitions_dir=def_dir, exercise_id='squat')
cfg.normalization       = NormalizationConfig(enabled=True)
cfg.phase_segmentation  = PhaseSegmentationConfig(enabled=True)
cfg.motion_attribution  = MotionAttributionConfig(enabled=True)

pipe_df, pipe_report = run_pipeline(df_raw, config=cfg, landmarks=LANDMARKS)

assert 'motion_attribution' in pipe_report
for col in expected_cols:
    assert col in pipe_df.columns
print(f'PASS: pipeline ⑦ motion_attribution present')
print(f'steps executed: {list(pipe_report.keys())}')

## Interpretation

| Check | Expected for bilateral_symmetric |
|---|---|
| `detected_active_limb` | all None |
| `expected_active_limb` | all None |
| `attribution_consistent` | all None |
| `attribution_action` | all None |
| Coordinates | unchanged |

**Alternating exercises** (lunge, plank_shoulder_tap): `detected_active_limb` will be
`'left'` or `'right'` per rep, and `attribution_consistent` will flag mismatches.
Test with the corresponding synthetic data once available.